In [1]:
import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score
)

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
import io

with open("../data/raw/original_RT_IoT2022.csv", "r", encoding="utf-8") as f:
    text = f.read()

# Remove quotation marks
text = text.replace('"', '')

# Fix leading comma in header if present
lines = text.splitlines()

if lines[0].startswith(","):
    lines[0] = lines[0][1:]

text = "\n".join(lines)

# Load dataset
df = pd.read_csv(io.StringIO(text))

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'original_RT_IoT2022.csv'

In [ ]:
cols_to_drop = [
    "Unnamed: 0",
    "Flow_ID",
    "Source_IP",
    "Destination_IP",
    "Timestamp",
    "id.orig_p",
    "id.resp_p"
]

df = df.drop(
    columns=[c for c in cols_to_drop if c in df.columns],
    errors="ignore"
)

print("Shape after removing unnecessary columns:", df.shape)

In [ ]:
feat_cols_for_dedup = [
    c for c in df.columns
    if c != "Attack_type"
]

before = len(df)

df = (
    df.drop_duplicates(subset=feat_cols_for_dedup)
      .reset_index(drop=True)
)

after = len(df)

print("=" * 60)
print("DATASET DEDUPLICATION")
print("=" * 60)
print("Rows before deduplication :", before)
print("Rows after deduplication  :", after)
print("Duplicate rows removed    :", before - after)

In [ ]:
normal_traffic = [
    "Thing_Speak",
    "Wipro_bulb",
    "MQTT_Publish"
]

if df["Attack_type"].dtype == object:

    df["Attack_type"] = df["Attack_type"].apply(
        lambda x: 0 if x in normal_traffic else 1
    )

    print("Attack_type converted to binary.")

else:
    print("Attack_type is already numeric.")

In [ ]:
label_encoders = {}

for col in df.select_dtypes(include=["object"]).columns:

    if col == "Attack_type":
        continue

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    label_encoders[col] = le

X = df.drop(columns=["Attack_type"])
y = df["Attack_type"]

print("=" * 60)
print("FEATURE / TARGET INFORMATION")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nNumber of classes:", y.nunique())

In [ ]:
# ---------------------------------------------------------
# STEP 1: 80% temporary training pool + 20% final test
# ---------------------------------------------------------

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# ---------------------------------------------------------
# STEP 2: Split remaining 80% into:
# 75% of 80% = 60% total training
# 25% of 80% = 20% total validation
# ---------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)

print(f"Total samples      : {len(X)}")
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")

print("\nPercentages:")
print(f"Training   : {len(X_train)/len(X)*100:.2f}%")
print(f"Validation : {len(X_val)/len(X)*100:.2f}%")
print(f"Test       : {len(X_test)/len(X)*100:.2f}%")

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")
print("Training scaled shape  :", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)
print("Test scaled shape      :", X_test_scaled.shape)

In [ ]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ),
    param_distributions=rf_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Random Forest...")
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("\nBest RF parameters:")
print(rf_search.best_params_)

In [ ]:
xgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=xgb_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_

print("\nBest XGBoost parameters:")
print(xgb_search.best_params_)

In [ ]:
lgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 7, 9, -1],
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "min_child_samples": [5, 10, 20]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(
        verbose=-1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=lgb_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training LightGBM...")
lgb_search.fit(X_train, y_train)

best_lgb = lgb_search.best_estimator_

print("\nBest LightGBM parameters:")
print(lgb_search.best_params_)

In [ ]:
# ============================================================
# SVM BASELINE MODEL
# SVM is used only as a conventional baseline.
# It is NOT included in DA-AMS model selection.
# ============================================================

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import time
import joblib
import os

print("Training SVM baseline...")

# ------------------------------------------------------------
# 1. Initialize SVM with fixed, conventional parameters
# ------------------------------------------------------------

svm_baseline = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=False,
    cache_size=2048,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 2. Train SVM
# ------------------------------------------------------------

svm_start = time.perf_counter()

svm_baseline.fit(X_train_scaled, y_train)

svm_train_time = time.perf_counter() - svm_start

print(f"SVM training completed in {svm_train_time:.2f} seconds.")

# ------------------------------------------------------------
# 3. Prediction
# ------------------------------------------------------------

svm_pred_start = time.perf_counter()

svm_pred = svm_baseline.predict(X_test_scaled)

svm_pred_time = time.perf_counter() - svm_pred_start

# ------------------------------------------------------------
# 4. Predictive performance
# ------------------------------------------------------------

svm_accuracy = accuracy_score(y_test, svm_pred)

svm_macro_f1 = f1_score(
    y_test,
    svm_pred,
    average="macro"
)

# decision_function is used because probability=False
svm_decision = svm_baseline.decision_function(X_test_scaled)

svm_roc_auc = roc_auc_score(
    y_test,
    svm_decision
)

# ------------------------------------------------------------
# 5. Per-flow inference latency
# ------------------------------------------------------------

svm_num_flows = len(X_test_scaled)

svm_latency_per_flow = (
    svm_pred_time / svm_num_flows
)

# ------------------------------------------------------------
# 6. Model size
# ------------------------------------------------------------

svm_model_path = "svm_baseline.pkl"

joblib.dump(
    svm_baseline,
    svm_model_path
)

svm_model_size_mb = (
    os.path.getsize(svm_model_path) / (1024 ** 2)
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("SVM BASELINE RESULTS")
print("=" * 55)

print(f"Accuracy              : {svm_accuracy:.6f}")
print(f"Macro-F1              : {svm_macro_f1:.6f}")
print(f"ROC-AUC               : {svm_roc_auc:.6f}")
print(f"Training Time (s)     : {svm_train_time:.4f}")
print(f"Total Inference (s)   : {svm_pred_time:.6f}")
print(f"Latency / Flow (s)    : {svm_latency_per_flow:.9f}")
print(f"Model Size (MB)       : {svm_model_size_mb:.6f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        svm_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

In [ ]:
# ============================================================
# DA-AMS CANDIDATE MODEL EVALUATION
# SVM is NOT included because it is a separate baseline.
# ============================================================

candidate_models = {
    "Random Forest": (best_rf, X_val),
    "XGBoost": (best_xgb, X_val),
    "LightGBM": (best_lgb, X_val)
}

validation_results = []

print("=" * 70)
print("DA-AMS CANDIDATE MODEL EVALUATION")
print("=" * 70)

for model_name, (model, X_eval) in candidate_models.items():

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    start_time = time.perf_counter()

    y_pred = model.predict(X_eval)

    end_time = time.perf_counter()

    # Number of flows
    n_flows = len(X_eval)

    # Per-flow inference latency
    latency = (end_time - start_time) / n_flows

    # --------------------------------------------------------
    # Predictive metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro"
    )

    # --------------------------------------------------------
    # Correct predictions and errors
    # --------------------------------------------------------

    correct_predictions = int(
        (y_pred == y_val).sum()
    )

    errors = int(
        (y_pred != y_val).sum()
    )

    # --------------------------------------------------------
    # ROC-AUC
    # --------------------------------------------------------

    if hasattr(model, "predict_proba"):

        y_prob = model.predict_proba(X_eval)[:, 1]

        roc_auc = roc_auc_score(
            y_val,
            y_prob
        )

    else:

        y_score = model.decision_function(X_eval)

        roc_auc = roc_auc_score(
            y_val,
            y_score
        )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    validation_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "MacroF1": macro_f1,
        "ROC_AUC": roc_auc,
        "Correct": correct_predictions,
        "Errors": errors,
        "Latency": latency
    })

    # --------------------------------------------------------
    # Display individual results
    # --------------------------------------------------------

    print(f"\n{model_name}")
    print(f"Accuracy          : {accuracy:.6f}")
    print(f"Macro-F1          : {macro_f1:.6f}")
    print(f"ROC-AUC           : {roc_auc:.6f}")
    print(f"Correct           : {correct_predictions}")
    print(f"Errors            : {errors}")
    print(f"Latency/flow      : {latency:.9f} s")


# ============================================================
# Convert results to DataFrame
# ============================================================

validation_df = pd.DataFrame(validation_results)

print("\nDA-AMS candidate evaluation completed.")

In [ ]:
display_df = validation_df.copy()

display_df["Accuracy"] = (
    display_df["Accuracy"] * 100
)

display_df["MacroF1"] = (
    display_df["MacroF1"] * 100
)

display_df["ROC_AUC"] = (
    display_df["ROC_AUC"] * 100
)

display_df

In [ ]:
import joblib
import os

# ============================================================
# SERIALIZED MODEL SIZE
# ============================================================

models_for_size = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb
}
model_sizes = {}

for model_name, model in models_for_size.items():

    filename = model_name.replace(" ", "_").lower() + "_size.pkl"

    joblib.dump(model, filename)

    size_mb = os.path.getsize(filename) / (1024 ** 2)

    model_sizes[model_name] = size_mb

    print(f"{model_name}: {size_mb:.6f} MB")

# Attach ModelSize to validation_df
validation_df["ModelSize"] = validation_df["Model"].map(model_sizes)

In [ ]:
selection = validation_df.copy()

if "ModelSize" not in selection.columns:
    selection["ModelSize"] = selection["Model"].map(model_sizes)

# ---------------------------------------------------------
# Accuracy — Benefit criterion
# Higher is better
# ---------------------------------------------------------

acc_min = selection["Accuracy"].min()
acc_max = selection["Accuracy"].max()

if acc_max == acc_min:
    selection["AccScore"] = 1.0
else:
    selection["AccScore"] = (
        (selection["Accuracy"] - acc_min)
        / (acc_max - acc_min)
    )


# ---------------------------------------------------------
# Macro-F1 — Benefit criterion
# Higher is better
# ---------------------------------------------------------

f1_min = selection["MacroF1"].min()
f1_max = selection["MacroF1"].max()

if f1_max == f1_min:
    selection["F1Score"] = 1.0
else:
    selection["F1Score"] = (
        (selection["MacroF1"] - f1_min)
        / (f1_max - f1_min)
    )


# ---------------------------------------------------------
# Latency — Cost criterion
# Lower is better
# ---------------------------------------------------------

lat_min = selection["Latency"].min()
lat_max = selection["Latency"].max()

if lat_max == lat_min:
    selection["LatencyScore"] = 1.0
else:
    selection["LatencyScore"] = (
        (lat_max - selection["Latency"])
        / (lat_max - lat_min)
    )


# ---------------------------------------------------------
# Model Size — Cost criterion
# Lower is better
# ---------------------------------------------------------

size_min = selection["ModelSize"].min()
size_max = selection["ModelSize"].max()

if size_max == size_min:
    selection["MemoryScore"] = 1.0
else:
    selection["MemoryScore"] = (
        (size_max - selection["ModelSize"])
        / (size_max - size_min)
    )


print("Normalization completed.")

In [ ]:
# Deployment-priority weighting vector
W_ACCURACY = 0.40
W_F1 = 0.30
W_LATENCY = 0.20
W_MODEL_SIZE = 0.10

selection["DeploymentScore"] = (
      W_ACCURACY * selection["AccScore"]
    + W_F1 * selection["F1Score"]
    + W_LATENCY * selection["LatencyScore"]
    + W_MODEL_SIZE * selection["MemoryScore"]
)

selection = selection.sort_values(
    by="DeploymentScore",
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("DA-AMS MODEL RANKING")
print("=" * 80)

print(
    selection[
        [
            "Model",
            "Accuracy",
            "MacroF1",
            "Latency",
            "ModelSize",
            "AccScore",
            "F1Score",
            "LatencyScore",
            "MemoryScore",
            "DeploymentScore"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# DA-AMS SELECTED MODEL DEPLOYMENT
# SVM is a separate baseline and is NOT part of DA-AMS.
# ============================================================

model_map = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb
}

# Get the automatically selected model
best_model_name = selection.iloc[0]["Model"]
best_model = model_map[best_model_name]
best_deployment_score = (
    selection.iloc[0]["DeploymentScore"]
)

# Save selected model
joblib.dump(
    best_model,
    "Selected_IDPS_Model.pkl"
)

# Evaluate selected model on test set
test_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)
test_macro_f1 = f1_score(y_test, test_pred, average="macro")

print("=" * 70)
print("DEPLOYMENT-AWARE AUTOMATIC MODEL SELECTION")
print("=" * 70)

print(
    "Selected Detection Engine :",
    best_model_name
)

print(
    "Deployment Score          :",
    round(best_deployment_score, 4)
)

print(
    "Test Set Accuracy         :",
    round(test_accuracy, 6)
)

print(
    "Test Set Macro-F1         :",
    round(test_macro_f1, 6)
)

print(
    "Model saved as            :",
    "Selected_IDPS_Model.pkl"
)

print("=" * 70)

In [ ]:
cm = confusion_matrix(
    y_test,
    test_pred
)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))

plt.imshow(
    cm,
    interpolation="nearest",
    cmap=plt.cm.Blues
)

plt.title(
    f"Confusion Matrix - {best_model_name}"
)

plt.colorbar()

plt.xticks(
    [0, 1],
    ["Normal", "Attack"]
)

plt.yticks(
    [0, 1],
    ["Normal", "Attack"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black"
        )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for name, (model, X_eval) in candidate_models.items():

    probs = model.predict_proba(
        X_eval
    )[:, 1]

    fpr, tpr, _ = roc_curve(
        y_val,
        probs
    )

    roc_auc_value = auc(
        fpr,
        tpr
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={roc_auc_value:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curve Comparison of Candidate Detection Models"
)

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):

    importance = pd.Series(
        best_model.feature_importances_,
        index=X.columns
    )

    importance = (
        importance
        .sort_values(ascending=False)
        .head(15)
        .sort_values()
    )

    plt.figure(figsize=(8, 6))

    importance.plot(
        kind="barh"
    )

    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")

    plt.title(
        f"Top 15 Most Important Features - {best_model_name}"
    )

    plt.tight_layout()
    plt.show()

else:

    print(
        f"Selected model is {best_model_name}; "
        "feature importance is not applicable."
    )

In [ ]:
# Reload original dataset for category-based visualization

traffic_df = pd.read_csv(
    io.StringIO(text)
)

selected_categories = [
    "MQTT_Publish",
    "Thing_Speak",
    "DOS_SYN_Hping"
]

selected_metrics = [
    "flow_duration",
    "flow_pkts_per_sec",
    "payload_bytes_per_second"
]

available_categories = [
    c for c in selected_categories
    if c in traffic_df["Attack_type"].unique()
]

available_metrics = [
    c for c in selected_metrics
    if c in traffic_df.columns
]

mean_values = (
    traffic_df[
        traffic_df["Attack_type"].isin(
            available_categories
        )
    ]
    .groupby("Attack_type")[available_metrics]
    .mean()
)

mean_values

mean_values.plot(
    kind="bar",
    figsize=(9, 6),
    logy=True
)

plt.xlabel("Traffic Category")
plt.ylabel("Mean Value (log scale)")

plt.title(
    "Comparison of Mean Network-Flow Characteristics"
)

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

In [ ]:
demo_df = X_val.copy()

demo_df["Target"] = y_val.values

normal_baseline = (
    demo_df[
        demo_df["Target"] == 0
    ]
    .drop(columns=["Target"])
    .mean()
)

normal_samples = (
    demo_df[
        demo_df["Target"] == 0
    ]
    .sample(
        min(7, len(demo_df[demo_df["Target"] == 0])),
        random_state=42
    )
)

attack_samples = (
    demo_df[
        demo_df["Target"] == 1
    ]
    .sample(
        min(7, len(demo_df[demo_df["Target"] == 1])),
        random_state=42
    )
)

demo_samples = (
    pd.concat(
        [
            normal_samples,
            attack_samples
        ]
    )
    .sample(
        frac=1,
        random_state=42
    )
)

print(
    "Normal samples selected :",
    len(normal_samples)
)

print(
    "Attack samples selected :",
    len(attack_samples)
)

print(
    "Total demonstration flows:",
    len(demo_samples)
)

In [ ]:
def detect_and_explain(
    raw_flow_df,
    model,
    baseline,
    actual_label
):

    start = time.perf_counter()

    prediction = model.predict(
        raw_flow_df
    )[0]

    if hasattr(model, "predict_proba"):

        probability = model.predict_proba(
            raw_flow_df
        )[0, 1]

    else:

        probability = None

    inference_time = (
        time.perf_counter() - start
    ) * 1000

    print("=" * 70)

    if prediction == 1:

        print("DETECTION RESULT : ATTACK DETECTED")
        if probability is not None:
            print(f"Attack probability : {probability:.4f}")

    else:

        print("DETECTION RESULT : NORMAL")
        if probability is not None:
            print(f"Attack probability : {probability:.4f}")

    print(
        f"Actual label       : {actual_label}"
    )

    print(
        f"Inference time     : {inference_time:.4f} ms"
    )

    print("\nKey Abnormal Network Features (Top 3 Deviations vs Normal Baseline):")
    raw_flow = raw_flow_df.iloc[0]
    irregularities = []

    for feature in raw_flow.index:
        current = float(raw_flow[feature])
        normal = float(baseline[feature])

        if normal != 0 and np.isfinite(normal):
            deviation = abs(current - normal) / abs(normal)
        else:
            deviation = abs(current)

        if not np.isfinite(deviation):
            deviation = 0.0

        irregularities.append((feature, current, normal, deviation))

    irregularities.sort(key=lambda x: x[3], reverse=True)

    for feat, curr, norm, dev in irregularities[:3]:
        print(f" \u2022 {feat}")
        print(f"      Current : {curr:.4f}")
        print(f"      Normal  : {norm:.4f}")

    print("=" * 70)

    return prediction

In [ ]:
# ============================================================
# REAL-TIME IDPS DEMONSTRATION & FLOW SCANNING
# ============================================================

print("=" * 80)
print("        LIGHTWEIGHT REAL-TIME IoT IDPS MONITORING")
print("=" * 80)
print(f"Selected Detection Engine : {best_model_name}")
print(f"Deployment Score          : {selection.iloc[0]['DeploymentScore']:.4f}")
print("Status                    : Monitoring Network Traffic")
print("=" * 80)

successful_flows = 0

for i, (idx, row) in enumerate(demo_samples.iterrows()):

    print(f"\n================ FLOW {i+1} OF {len(demo_samples)} ================")

    actual_label = "Attack" if row["Target"] == 1 else "Normal"
    raw_flow_df = row.drop("Target").to_frame().T

    try:
        detect_and_explain(
            raw_flow_df=raw_flow_df,
            model=best_model,
            baseline=normal_baseline,
            actual_label=actual_label
        )
        print(f"FLOW {i+1} Processed Successfully")
        successful_flows += 1

    except Exception as e:
        import traceback
        print(f"ERROR in FLOW {i+1}: {e}")
        traceback.print_exc()

print("\n" + "=" * 80)
print(f"Monitoring Completed -- {successful_flows}/{len(demo_samples)} flows processed successfully")
print("=" * 80)